# Домашнее задание №2: Медицинская сегментация изображений

В этом задании вы реализуете полноценный пайплайн семантической сегментации медицинских изображений: подготовка данных, архитектура U-Net, обучение, оценка качества и интерпретация результатов.

## Оглавление
- [Правила игры и оценивание](#rules)
- [1. Подготовка окружения и импорты](#setup)
- [2. Выбор и загрузка датасета (Kaggle)](#data-download)
- [3. Теоретический минимум (Sanity Check)](#theory)
- [4. Разведочный анализ данных (EDA)](#eda)
- [5. Предобработка данных](#preprocessing)
- [6. Dataset и DataLoader](#dataloader)
- [7. Реализация метрик](#metrics)
- [8. Архитектура U-Net](#unet)
- [9. Функция потерь (Dice Loss)](#loss)
- [10. Обучение модели](#training)
- [11. Оценка результатов](#evaluation)
- [12. Интерпретация (Grad-CAM и др.)](#interpretability)
- [13. Бонус: Предобученные модели](#bonus)

---

<a id="rules"></a>
## Правила игры

Чтобы задание было полезным, а не превратилось в копипасту, придерживайтесь следующих принципов.

### 1. Использование AI‑ассистентов (теория и код)

Вы можете использовать LLM (ChatGPT, Claude и др., все-таки 26 год на дворе), но по понятным и прозрачным правилам:

- **Теоретические вопросы.**  
  Сначала вставьте «сырой» ответ от ИИ, а ниже обязательно напишите свой ответ, перефразировав и дополнив его. Без вашего текста блок не засчитывается (0 баллов). Так вы тренируетесь именно понимать материал, а не списывать :)
- **Код (Dataset, U‑Net, Loss, Train Loop и т.п.).**  
  Если используете подсказки ИИ, приложите:
  - краткое объяснение от ИИ: что делает этот фрагмент,
  - свои комментарии: почему он устроен именно так, какой смысл у решений, есть ли альтернативы.
  В обратном случае, можете указывать, что всё писали с нуля самостоятельно.
- По ДЗ могут задаваться вопросы на устной контрольной в рамках курса, поэтому нужно уметь объяснить написанное.

### 2. Самостоятельная реализация

- **Разрешено:** менять аргументы, переписывать логику под себя, добавлять свои проверки и фичи, улучшать структуру кода.
- **Нельзя:** решать ключевые части с помощью готовых высокоуровневых обёрток, например:
  - не импортируем готовую U‑Net из `segmentation_models_pytorch`,
  - не используем готовые `DiceLoss`/метрики из сторонних библиотек (если это не явно оговорено).
- Цель задания — научиться руками строить пайплайн и понимать, что внутри. Если большая часть реализована через готовые чёрные ящики, баллы могут быть сильно снижены, устремившись в пределе к нулю :(

### 3. Требования к качеству и система оценивания

Базовые требования к модели на валидации:

- Dice Score ≥ 0.60  
- IoU ≥ 0.50  

Если требования не выполнены, итоговый балл за практическую часть может быть снижено (на усмотрение проверяющего). 

| Раздел                     | Баллы | Раздел               | Баллы |
| :------------------------- | :---: | :------------------- | :---: |
| Теория (Вопросы 1–2)       |  1.0  | Архитектура U‑Net    |  1.5  |
| EDA (анализ данных)        |  1.0  | Функция потерь       |  1.0  |
| Предобработка              |  1.0  | Цикл обучения        |  1.5  |
| Dataset & DataLoader       |  1.0  | Оценка результатов   |  0.5  |
| Метрики                    |  1.0  | Интерпретация        |  0.5  |
| **Итого базовая часть**    | **10.0** | **Бонус (Pretrained)** | **+2.0** |

**Формат сдачи:** Отправьте этот `.ipynb` ноутбук со всеми выполненными ячейками, сохраненными аутпутами и текстовыми ответами в форму сдачи.


<a id="setup"></a>
# 1. Подготовка окружения и импорты 🛠️

В этом разделе мы устанавливаем недостающие библиотеки и импортируем всё необходимое для работы: PyTorch, аугментации, инструменты визуализации и т.д.

In [ ]:
# Установка необходимых библиотек
# Раскомментируйте нужные строки, если чего‑то не хватает

# !pip install torch torchvision
# !pip install albumentations
# !pip install kaggle
# !pip install mlflow  # или wandb
# !pip install matplotlib seaborn
# !pip install nibabel  # для BraTS (NIfTI файлы)
# !pip install opencv-python
# !pip install tqdm

In [ ]:
# Основные импорты

import os
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from tqdm import tqdm

warnings.filterwarnings("ignore")

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

# Аугментации
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Для BraTS (при необходимости)
# import nibabel as nib

# Логирование (по выбору)
# import mlflow
# import wandb

In [ ]:
# Фиксация random seed для воспроизводимости

def set_seed(seed: int = 42) -> None:
    """
    Устанавливает seed для основных генераторов случайных чисел.
    """
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Используемое устройство: {device}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"Доступная память: {mem_gb:.2f} GB")

<a id="data-download"></a>
# 2. Выбор и загрузка датасета (Kaggle)

Для практики выберите **один** из двух датасетов. Оба описывают реальные медицинские задачи:

1. [BraTS2020](https://www.kaggle.com/datasets/awsaf49/brats2020-training-data) — сегментация опухолей головного мозга на МРТ (сильно посложней, для каждого пациента 3–4 МРТ‑модальности (T1, T1ce, T2, FLAIR), которые нужно правильно нормировать, стэчить, иногда учитывать, что часть модальностей может быть «шумной» или отсутствовать)
2. [ISIC2018](https://www.kaggle.com/datasets/tschandl/isic2018-challenge-task1-data-segmentation) — сегментация кожных поражений на дерматоскопических изображениях (попроще, **рекомендуем**, если вы не проходите компьютерные игры на уровне "Бессмертный")


## 2.1 Настройка Kaggle API

Чтобы скачать данные прямо в ноутбук, нужен токен Kaggle API:

1. Откройте настройки профиля: https://www.kaggle.com/settings  
2. Нажмите **Create New API Token** — скачается файл `kaggle.json`.  
3. Поместите `kaggle.json` в нужную директорию.

### Размещение токена

**Linux / macOS:**
```bash
mkdir -p ~/.kaggle
mv ~/Downloads/kaggle.json ~/.kaggle/
chmod 600 ~/.kaggle/kaggle.json
```

**Для Windows:**
```bash
mkdir %USERPROFILE%\.kaggle
move %USERPROFILE%\Downloads\kaggle.json %USERPROFILE%\.kaggle\
```

**Для Google Colab:**
```python
from google.colab import files
files.upload()  # Загрузите kaggle.json
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
```

## Скачивание данных
Раскомментируйте ячейку с кодом для **выбранного вами** датасета и выполните её.

### Вариант А: BraTS2020 (МРТ мозга)
```bash
# !kaggle datasets download -d awsaf49/brats2020-training-data
# !unzip -q brats2020-training-data.zip -d ./data/brats2020/
# !ls -la ./data/brats2020/
```

### Вариант B: ISIC2018 (Поражения кожи)
```bash
# !kaggle datasets download -d tschandl/isic2018-challenge-task1-data-segmentation
# !unzip -q isic2018-challenge-task1-data-segmentation.zip -d ./data/isic2018/
# !ls -la ./data/isic2018/
```

## 2.3 Проверка, что данные на месте

In [ ]:
# TODO: Укажите путь к выбранному датасету
DATA_PATH = "./data/isic2018/"  # или "./data/brats2020/"

if os.path.exists(DATA_PATH):
    print(f"Датасет найден по пути: {DATA_PATH}")
    print("Содержимое каталога:")
    for item in os.listdir(DATA_PATH):
        print(f" - {item}")
else:
    print(f"Датасет не найден по пути: {DATA_PATH}")
    print("Проверьте инструкции по скачиванию выше.")

## Полезные ссылки

- [Официальная документация Kaggle API](https://github.com/Kaggle/kaggle-api)
- [Как использовать Kaggle API](https://www.kaggle.com/docs/api)
- [Kaggle API в Google Colab](https://www.kaggle.com/general/74235)
- [Troubleshooting Kaggle API](https://github.com/Kaggle/kaggle-api/issues)


<a id="theory"></a>
# 3. Теоретический минимум (Sanity Check)

### Вопрос 1. Что такое сегментация изображений?

> **Задание:**
>   - Кратко сравните задачи компьютерного зрения: классификация, детекция объектов и сегментация (семантическая и/или экземплярная). В чем ключевая разница в их *выходных данных*?
>   - Приведите 1–2 примера медицинских задач, где применяется каждый из этих подходов (не только сегментация).
>   - Объясните, почему сегментация, а не просто классификация наличия болезни, особенно важна в медицине для планирования лечения и мониторинга.
>   
> **Ответ ИИ‑ассистента:** [TODO: вставьте сюда сгенерированный ответ]
>
> **Ваш ответ своими словами:** [TODO: перефразируйте и допишите свои рассуждения]

### Вопрос 2. Какие метрики используют для сегментации и почему?

> **Задание:**
>   - Опишите метрики: Dice Score, IoU (Jaccard), Precision / Recall на уровне пикселей.
>   - Объясните, чем они отличаются.
>   -  Обсудите, когда какая‑то метрика может быть «обманчивой» (например, при сильном дисбалансе классов) и почему простая accuracy в сегментации часто бесполезна.
>   
> **Ответ ИИ‑ассистента:** [TODO: вставьте сюда сгенерированный ответ]
>
> **Ваш ответ своими словами:** [TODO: перефразируйте и допишите свои рассуждения]

> **Полезные материалы:**
> *   Отличная визуализация разницы между задачами CV: [Link](https://blogs.nvidia.com/wp-content/uploads/2016/08/object_detection_vs_segmentation.png) (нужен хороший релевантный пример, можно поискать на towardsdatascience)
> *   Глава "Computer Vision" из учебника [Dive into Deep Learning](https://d2l.ai/chapter_computer-vision/index.html) (доступна на русском).

<a id="eda"></a>
# 4. Исследование структуры датасета (EDA)

**Что должно быть сделано в этом разделе (Definition of done):**

Минимально ожидается, что вы:
>1. Визуализируете **5–10 примеров** из датасета: для каждого показаны:
>    - исходное изображение;
>    - ground truth маска;
>    - наложение маски на изображение (overlay / прозрачность).
> 2. Считаете и выводите базовую статистику:
>    - общее число изображений в train/val;
>    - типичный размер (H, W) изображений и масок;
>    - диапазон значений пикселей (мин/макс, при желании mean/std);
>    - распределение площади маски (доля пикселей класса 1 в процентах);
>    - доля **пустых масок** (где объект отсутствует).
>3. Делаете 1–2 простые визуализации статистики (например, гистограмма площади маски).
>4. Пишите короткий текстовый вывод:
>    - есть ли сильный дисбаланс (много пустых масок, очень маленькие объекты);
>    - какие артефакты/особенности изображений вы заметили (волосы, рамки, шумы и т.п.);
>    - как это может повлиять на метрики (почему, например, pixel accuracy может быть обманчивой).


## 4.1 Загрузка и визуализация примеров

In [ ]:
# ========== BraTS2020 ==========
def load_brats_sample(patient_path: Path, slice_idx: int | None = None):
    """
    Загрузка одного 2D‑среза из BraTS.

    Args:
        patient_path: путь к папке конкретного пациента.
        slice_idx: индекс среза (если None — взять средний).

    Returns:
        image: np.ndarray, один срез одной модальности (например, T2)
        mask:  np.ndarray, соответствующая маска
    """
    # TODO: реализуйте загрузку для BraTS
    # Подсказки:
    # 1. Используйте nibabel для чтения .nii.gz
    # 2. Выберите одну модальность (например, FLAIR)
    # 3. Возьмите один срез по оси глубины
    # 4. Загрузите *_seg.nii.gz как маску
    # 5. При необходимости сведите мультклассовую маску к бинарной
    raise NotImplementedError


# ========== ISIC2018 ==========
def load_isic_sample(image_path: Path, mask_path: Path):
    """
    Загрузка одного примера из ISIC.

    Args:
        image_path: путь к RGB‑изображению.
        mask_path:  путь к бинарной маске.

    Returns:
        image: np.ndarray, RGB изображение.
        mask:  np.ndarray, бинарная маска 0/1.
    """
    # TODO: реализуйте загрузку для ISIC
    # Подсказки:
    # 1. Используйте cv2 или PIL для чтения изображения и маски
    # 2. Преобразуйте маску в 0/1
    raise NotImplementedError


In [ ]:
def visualize_samples(images, masks, n_samples=5, figsize=(12, 4)):
    """
    Визуализация изображений и масок.

    Args:
        images: список np.ndarray или torch.Tensor изображений (H, W, C) или (C, H, W)
        masks: список np.ndarray или torch.Tensor масок (H, W) или (1, H, W)
        n_samples: количество примеров для визуализации
        figsize: размер фигуры matplotlib
    """
    n_samples = min(n_samples, len(images))
    plt.figure(figsize=(figsize[0], figsize[1] * n_samples))
    
    for i in range(n_samples):
        img = images[i]
        mask = masks[i]

        # Приводим к numpy и формату (H, W, C)
        if isinstance(img, torch.Tensor):
            img_np = img.detach().cpu().numpy()
        else:
            img_np = img

        if img_np.ndim == 3 and img_np.shape[0] in (1, 3):
            # (C, H, W) -> (H, W, C)
            img_np = np.transpose(img_np, (1, 2, 0))
        elif img_np.ndim == 2:
            img_np = img_np[..., None]

        if isinstance(mask, torch.Tensor):
            mask_np = mask.detach().cpu().numpy()
        else:
            mask_np = mask

        if mask_np.ndim == 3 and mask_np.shape[0] == 1:
            mask_np = mask_np[0]
        
        # Нормализуем для отображения
        img_disp = img_np.copy()
        img_disp = (img_disp - img_disp.min()) / (img_disp.max() - img_disp.min() + 1e-8)

        # Подготовим цветной overlay
        mask_disp = (mask_np > 0).astype(np.float32)
        overlay = img_disp.copy()
        if overlay.shape[-1] == 1:
            overlay = np.repeat(overlay, 3, axis=-1)
        overlay[..., 0] = np.maximum(overlay[..., 0], mask_disp)  # красный канал

        # 3 колонки: image / mask / overlay
        idx_row = i

        plt.subplot(n_samples, 3, 3 * idx_row + 1)
        plt.imshow(img_disp.squeeze(), cmap='gray' if img_disp.shape[-1] == 1 else None)
        plt.title(f'Image #{i}')
        plt.axis('off')

        plt.subplot(n_samples, 3, 3 * idx_row + 2)
        plt.imshow(mask_disp, cmap='gray')
        plt.title(f'Mask #{i}')
        plt.axis('off')

        plt.subplot(n_samples, 3, 3 * idx_row + 3)
        plt.imshow(overlay)
        plt.title(f'Overlay #{i}')
        plt.axis('off')

    plt.tight_layout()
    plt.show()


# TODO: Загрузите и визуализируйте 5-10 примеров из вашего датасета
# images = []
# masks = []

>  💡 **Подсказка:** Обратите внимание на контрастность изображений и четкость границ масок. Есть ли на изображениях артефакты (волосы, линейки, пузырьки воздуха для ISIC, или череп, артефакты МРТ для BraTS)? Это пригодится вам при анализе ошибок модели.

## 4.2 Анализ статистики датасета

In [ ]:
# TODO: Проанализируйте ваш датасет

def analyze_dataset(data_path: str | Path, dataset_type: str = "isic"):
    """
    Базовый анализ датасета.

    Вопросы, на которые стоит ответить:
    1. Сколько всего примеров?
    2. Какой типичный размер изображений?
    3. Диапазон значений пикселей?
    4. Каково соотношение классов (доля пикселей объекта)?
    5. Есть ли примеры с пустыми масками?
    """
    # TODO: реализуйте анализ
    # Можно собрать статистику по подвыборке, чтобы не перегружать ноутбук.
    raise NotImplementedError


# Пример вызова:
# analyze_dataset(DATA_PATH, dataset_type="isic")  # или "brats"

## 4.3 Биологическое / медицинское объяснение

### Вопрос 3.

> **Если вы используете BraTS:**
>   - Чем отличаются модальности MRI (T1, T1ce, T2, FLAIR)?
>   - Как опухоль выглядит на разных модальностях?
>   - Почему важно качественно сегментировать опухоль?
>   - Какие клинические решения опираются на сегментацию?
>   
> **Если вы используете ISIC:**
>   - Что такое дерматоскопические изображения?
>   - Какие визуальные признаки поражений кожи вы видите на примерах?
>   - Почему критична ранняя диагностика меланомы?
>   - Как сегментация помогает врачу в диагностике?
>   
> **Ответ ИИ‑ассистента:** [TODO: вставьте сюда сгенерированный ответ]
>
> **Ваш ответ своими словами:** [TODO: перефразируйте и допишите свои рассуждения]

<a id="preprocessing"></a>
# 5. Предобработка данных

### Вопрос 4.

> **Зачем нужна предобработка?**
> 
> **Ответ ИИ‑ассистента:** [TODO: вставьте сюда сгенерированный ответ]
>
> **Ваш ответ своими словами:** [TODO: объясните, почему нормализация, приведение размера и аугментации важны для обучения]

## 5.1 Нормализация интенсивностей

**Зачем это нужно?** Нейронные сети любят, когда входные данные имеют предсказуемое распределение (например, среднее 0, дисперсия 1). Это ускоряет и стабилизирует сходимость градиентного спуска. Пиксели медицинских изображений (особенно МРТ) могут иметь очень разный диапазон интенсивностей от снимка к снимку, что затрудняет обучение.

**Задача:** привести значения пикселей к стабильному диапазону (например, [0, 1] или распределение с нулевым средним и единичным стандартным отклонением).

На практике часто используют два подхода к нормализации:

> 1. **Глобальная нормализация по датасету**  
>    - Считаем среднее и стандартное отклонение по **train** выборке (отдельно по каналам).
>    - Для каждого изображения делаем: $x' = (x - \mu) / \sigma$.
>    - Плюсы: модель видит данные в согласованном масштабе, особенно полезно для RGB (ISIC).
>    - Минусы: нужно заранее один раз пройтись по датасету.

> 2. **Нормализация по каждому изображению отдельно**  
>    - Min-max: $x' = (x - x_{min}) / (x_{max} - x_{min})$.
>    - Z-score по изображению: вычитаем mean и делим на std конкретного снимка.
>    - Плюсы: просто реализовать.
>    - Минусы: для некоторых модальностей (MRI) может «маскировать» физический смысл интенсивностей.

In [ ]:
def normalize_image(image: np.ndarray, method: str = "minmax") -> np.ndarray:
    """
    Нормализация изображения.

    Args:
        image: входное изображение (np.ndarray).
        method: 'minmax' или 'zscore'.

    Returns:
        normalized_image: нормализованное изображение.
    """
    # TODO: Реализуйте нормализацию
    # 
    # Метод 1: Min-Max нормализация (0-1)
    #
    # Метод 2: Z-score нормализация (mean=0, std=1)
    #
    # Выберите подходящий метод для вашего датасета
    

# Тестирование
# sample_image = np.random.rand(256, 256)
# normalized = normalize_image(sample_image, method='')
# print(f'Исходный диапазон: [{sample_image.min():.3f}, {sample_image.max():.3f}]')
# print(f'После нормализации: [{normalized.min():.3f}, {normalized.max():.3f}]')

In [ ]:
# ===== SANITY CHECK: normalize_image =====
test_img = np.array([[0.0, 50.0], [100.0, 200.0]])

norm_mm = normalize_image(test_img, method='minmax')
assert abs(float(norm_mm.min()) - 0.0) < 1e-5, "minmax: min должен быть 0"
assert abs(float(norm_mm.max()) - 1.0) < 1e-5, "minmax: max должен быть 1"

norm_z = normalize_image(test_img, method='zscore')
assert abs(float(norm_z.mean())) < 1e-4, "zscore: среднее должно быть ≈0"
assert abs(float(norm_z.std()) - 1.0) < 1e-3, "zscore: std должно быть ≈1"

print("✅ normalize_image: все проверки пройдены!")

### Вопрос 5. 
> **Какой метод нормализации выбрать и почему?**
> 
> **Ответ ИИ‑ассистента:** [TODO: вставьте ответ]
> 
> **Ваш ответ своими словами:** [TODO: аргументируйте выбор для вашего датасета (BraTS или ISIC)]

## 5.2 Изменение размера (resize)

**Задача:** привести все изображения и маски к единому размеру, чтобы их можно было батчить.

Важно: для **изображения** и **маски** нужно использовать разные режимы интерполяции:

- Изображение: `INTER_LINEAR` (или `INTER_AREA`) — плавное изменение, сглаживание.
- Маска (дискретные классы): `INTER_NEAREST` — ближайший сосед, чтобы не появлялись промежуточные значения между классами.

In [ ]:
def resize_image_and_mask(
    image: np.ndarray,
    mask: np.ndarray,
    target_size: tuple[int, int] = (256, 256),
) -> tuple[np.ndarray, np.ndarray]:
    """
    Изменение размера изображения и маски.

    Args:
        image: исходное изображение.
        mask: исходная маска.
        target_size: (height, width).

    Returns:
        resized_image, resized_mask.
    """
    # TODO: Реализуйте resize
    # 
    # Важно: не забудьте про интерполяцию для image и mask (они не могут совпадать!)
    #
    # Пример с OpenCV:
    # resized_image = cv2.resize(image, target_size, interpolation=cv2.INTER_LINEAR)
    # resized_mask = cv2.resize(mask, target_size, interpolation=cv2.INTER_NEAREST)
    


In [ ]:
# ===== SANITY CHECK: resize_image_and_mask =====
img_test  = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
mask_test = np.random.randint(0, 2,   (480, 640),    dtype=np.uint8)

r_img, r_mask = resize_image_and_mask(img_test, mask_test, target_size=(256, 256))

assert r_img.shape[:2]  == (256, 256), f"Размер изображения неверный: {r_img.shape}"
assert r_mask.shape[:2] == (256, 256), f"Размер маски неверный: {r_mask.shape}"
assert set(np.unique(r_mask)).issubset({0, 1}), \
    f"Маска содержит значения кроме 0 и 1: {np.unique(r_mask)} — используйте INTER_NEAREST!"

print("✅ resize_image_and_mask: все проверки пройдены!")

### Вопрос 6

> **Почему для маски важен выбор интерполяции?**
>   - Объясните, почему для маски нельзя использовать, например, `INTER_LINEAR`.
>   - Что происходит с границами и классами при неправильной интерполяции?
>   
> **Ответ ИИ‑ассистента:** [TODO]
>
>**Ваш ответ своими словами:** [TODO]

## 5.3 Аугментация данных

**Цель:** увеличить разнообразие обучающей выборки и снизить переобучение, сохраняя при этом медицинский смысл изображений.

In [ ]:
import albumentations as A
from albumentations.pytorch import ToTensorV2

def get_training_augmentation():
    """
    Аугментации для обучающей выборки.
    Подбирайте аккуратно: важно не искажать медицинскую реальность.
    """
    train_transform = A.Compose([
        # TODO: Добавьте аугментации
        # Обязательно в конце:
        # A.Normalize(mean=..., std=...),
        # ToTensorV2()
    ])
    return train_transform

def get_validation_augmentation():
    """
    Аугментации для валидации: как правило, только приведение к формату модели.
    """
    val_transform = A.Compose(
        # TODO: Только нормализация и преобразование в тензор
        # A.Normalize(mean=..., std=...),
        # ToTensorV2()
    ])
    return val_transform

### Вопрос 7

> **Какие аугментации подходят для медицинских изображений?**
>   - Какие аугментации вы выбрали и почему?
>   - Какие трансформации лучше не использовать (или использовать очень осторожно), чтобы не сломать клинический смысл?
>   - В чем ключевое отличие библиотеки `albumentations` от модуля `torchvision.transforms`? (Подсказка: подумайте об эффективности и о том, как они применяются к изображению и маске одновременно).
>
> **Ответ ИИ‑ассистента:** [TODO]
>
> **Ваш ответ своими словами:** [TODO]


> **Полезные материалы:**
> *   [Albumentations Documentation](https://albumentations.ai/docs/) - отличная документация с примерами визуализации каждой аугментации.
> *   [PyTorch Torchvision Transforms](https://pytorch.org/vision/stable/transforms.html)

<a id="dataloader"></a>
# 6. Создание Dataset и DataLoader

В этом разделе вы реализуете класс `Dataset`, который:

- умеет находить пары (изображение, маска) в вашей структуре папок;
- загружает данные с диска;
- применяет предобработку и аугментации;
- возвращает тензоры в формате, удобном для модели.

После этого вы создаёте `DataLoader` для обучающей и валидационной выборок.

**Рекомендация:** явно опишите в Markdown, как именно у вас устроены пути к данным (какие подпапки, как сопоставляются имена файлов).

## 6.1 Структура папок

**Пример структуры для ISIC:**

```text
data/
  isic2018/
    train/
      images/
        ISIC_0000000.jpg
        ISIC_0000001.jpg
        ...
      masks/
        ISIC_0000000_segmentation.png
        ISIC_0000001_segmentation.png
        ...
    val/
      images/
        ...
      masks/
        ...
```

**Пример структуры для BraTS:**

```
data/
  brats2020/
    train/
      BraTS20_Training_001/
        BraTS20_Training_001_flair.nii.gz
        BraTS20_Training_001_seg.nii.gz
        ...
    val/
      ...
```

> ⚠️ Важное замечание по памяти:
>   - При загрузке данных следите за тем, чтобы не загрузить сразу всё в RAM.
>   - Dataset должен загружать изображения по одному в __getitem__, а не хранить все в памяти.
>   - Если датасет маленький и помещается в RAM, можно загрузить всё сразу для ускорения, но это не рекомендуется для production-кода.

In [2]:
from torch.utils.data import Dataset
import os
from pathlib import Path

class MedicalSegmentationDataset(Dataset):
    """
    Обобщённый Dataset для задач медицинской сегментации.

    Поддерживает как BraTS, так и ISIC (в зависимости от dataset_type).
    """
    def __init__(self, data_path, transform=None, dataset_type='isic'):
        """
        Args:
            data_path: путь к данным
            transform: albumentations трансформации
            dataset_type: 'brats' или 'isic'
        """
        self.data_path = Path(data_path)
        self.transform = transform
        self.dataset_type = dataset_type
        
        # TODO: Соберите список путей к изображениям и маскам
        self.image_paths = []
        self.mask_paths = []
        
        if dataset_type == 'brats':
            # TODO: Реализуйте для BraTS
            # TODO: обойти папки пациентов и собрать пары (image, mask)

            pass
        elif dataset_type == 'isic':
            # TODO: Реализуйте для ISIC
            # TODO: собрать список файлов изображений и масок

            pass
        
        print(f'Найдено {len(self.image_paths)} примеров')
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx: int):
        """
        Возвращает один пример (image, mask) в формате тензоров.

        image: torch.Tensor, shape (C, H, W)
        mask:  torch.Tensor, shape (1, H, W)
        """
        # TODO: Загрузите изображение и маску
        
        if self.dataset_type == 'brats':
            # TODO: 
            pass
        elif self.dataset_type == 'isic':
            # TODO: 
            pass
        
        # TODO: Загрузите маску аналогично
        
        # Применение трансформаций
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
        
        # Убедитесь, что маска имеет правильную размерность
        if len(mask.shape) == 2:
            mask = mask.unsqueeze(0)  # (H, W) -> (1, H, W)
        
        return image, mask

In [ ]:
from torch.utils.data import DataLoader

# Параметры - пример
BATCH_SIZE = 8
NUM_WORKERS = 4  # Количество процессов для загрузки данных 

# Создание датасетов
train_dataset = MedicalSegmentationDataset(
    data_path='./data/your_dataset/train',
    transform=get_training_augmentation(),
    dataset_type='isic'  # или 'brats'
)

val_dataset = MedicalSegmentationDataset(
    data_path='./data/your_dataset/val',
    transform=get_validation_augmentation(),
    dataset_type='isic'  # или 'brats'
)

# Создание DataLoader
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True if torch.cuda.is_available() else False
)

In [ ]:
# ===== SANITY CHECK: Dataset =====
assert len(train_dataset) > 0, "train_dataset пустой!"
assert len(val_dataset)   > 0, "val_dataset пустой!"

img, mask = train_dataset[0]

assert isinstance(img,  torch.Tensor), f"image должен быть torch.Tensor, получили {type(img)}"
assert isinstance(mask, torch.Tensor), f"mask должен быть torch.Tensor, получили {type(mask)}"
assert img.ndim  == 3, f"image должен быть (C,H,W), получили shape={img.shape}"
assert mask.ndim == 3, f"mask должен быть (1,H,W), получили shape={mask.shape}"
assert mask.shape[0] == 1, f"Первый канал маски должен быть 1, получили {mask.shape[0]}"
assert img.shape[1] == mask.shape[1] and img.shape[2] == mask.shape[2], \
    f"H,W изображения {img.shape} и маски {mask.shape} не совпадают"
assert float(mask.min()) >= 0 and float(mask.max()) <= 1, \
    f"Маска должна быть в [0,1], получили [{mask.min()}, {mask.max()}]"

print(f"✅ Dataset: image={tuple(img.shape)}, mask={tuple(mask.shape)}, "
      f"train={len(train_dataset)}, val={len(val_dataset)}")

In [ ]:
sample_images = []
sample_masks = []

for i in range(10):
    img, mask = train_dataset[i]  # img: tensor (C, H, W), mask: tensor (1, H, W)
    sample_images.append(img)
    sample_masks.append(mask)

visualize_samples(sample_images, sample_masks, n_samples=5)

<a id="metrics"></a>
# 7. Метрики для оценки сегментации

Здесь вы реализуете базовые метрики качества сегментации:

- Dice Score,
- IoU (Jaccard),
- Pixel Accuracy,
- Precision и Recall на уровне пикселей.

Важно реализовать их «вручную», чтобы лучше понимать, как они устроены, и уметь интерпретировать результаты.

### 💡 Как работают метрики сегментации?

Все метрики сегментации строятся на четырех базовых показателях:
- **True Positives (TP):** пиксели, которые и модель, и ground truth считают объектом ✅
- **True Negatives (TN):** пиксели, которые и модель, и ground truth считают фоном ✅
- **False Positives (FP):** пиксели, которые модель ошибочно посчитала объектом (ложная тревога) ❌
- **False Negatives (FN):** пиксели, которые модель пропустила (объект есть, но модель его не видит) ❌

### Вопрос 8. 

> **Приведите примеры задач сегментации из мира и медицины. Какая вам лично интересней всего? Какие лаборатории / компании активней других ее исследуют?**
>
> **Ответ ИИ-ассистента:** [TODO: Вставьте ответ]
>
> **Ваш ответ своими словами:** [TODO: Перефразируйте]

## 7.1 Dice Score (Коэффициент Дайса)

**Объяснение:**
> - Dice Score измеряет overlap (перекрытие) между предсказанием и ground truth
> - Значения от 0 (нет перекрытия) до 1 (полное совпадение)
> - `smooth` параметр предотвращает деление на ноль при пустых масках
> - Умножение на 2 в числителе нормализует метрику

$$
\text{Dice} = \frac{2 \times |X \cap Y|}{|X| + |Y|} = \frac{2 \times TP}{2 \times TP + FP + FN}
$$

Где:
- $X$ - предсказанная маска
- $Y$ - ground truth маска
- $|X \cap Y|$ - пересечение (количество правильно предсказанных пикселей)
- $|X|$ - количество пикселей в предсказании
- $|Y|$ - количество пикселей в ground truth
- $TP$ - True Positives (правильно предсказанные положительные пиксели)
- $FP$ - False Positives (ложно предсказанные положительные пиксели)
- $FN$ - False Negatives (пропущенные положительные пиксели)

In [ ]:
def dice_score(pred_mask: torch.Tensor, true_mask: torch.Tensor, smooth: float = 1e-6) -> float:
    """
    Dice Score для бинарной сегментации.

    Args:
        pred_mask: предсказанная маска (0/1), shape (B, 1, H, W) или (B, H, W).
        true_mask: истинная маска (0/1), того же размера.
        smooth: сглаживание для численной стабильности.

    Returns:
        dice: скаляр (float).
    """

    # TODO
    
    pass

## 7.2 IoU (Intersection over Union / Jaccard Index) 

$$
\text{IoU} = \frac{|X \cap Y|}{|X \cup Y|} = \frac{TP}{TP + FP + FN}
$$

Где:
- $|X \cap Y|$ - пересечение (intersection)
- $|X \cup Y|$ - объединение (union)
- $TP$ - True Positives
- $FP$ - False Positives
- $FN$ - False Negatives

In [ ]:
def iou_score(pred_mask, true_mask, smooth=1e-6):
    """
    Вычисление IoU (Intersection over Union) для бинарной сегментации
    
    Args:
        pred_mask: предсказанная маска (torch.Tensor)
        true_mask: ground truth маска (torch.Tensor)
        smooth: сглаживающий коэффициент
    
    Returns:
        iou: IoU Score (float)
    """
    # TODO: Реализуйте вычисление IoU по аналогии с Dice Score
    
    pass

## 7.3 Pixel Accuracy 


$$
\text{Pixel Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
$$

In [ ]:
def pixel_accuracy(pred_mask, true_mask):
    """
    Вычисление точности на уровне пикселей
    
    Args:
        pred_mask: предсказанная маска (torch.Tensor)
        true_mask: ground truth маска (torch.Tensor)
    
    Returns:
        accuracy: Pixel Accuracy (float)
    """
    # TODO: Реализуйте вычисление pixel accuracy
    pass

## 7.4 Precision и Recall


$$
\text{Precision} = \frac{TP}{TP + FP}
$$

$$
\text{Recall} = \frac{TP}{TP + FN}
$$

In [ ]:
def pixel_precision(pred_mask, true_mask, smooth=1e-6):
    """
    Вычисление Precision на уровне пикселей
    
    Args:
        pred_mask: предсказанная маска (torch.Tensor)
        true_mask: ground truth маска (torch.Tensor)
        smooth: сглаживающий коэффициент
    
    Returns:
        precision: Precision (float)
    """
    # TODO: Реализуйте Precision
    pass

def pixel_recall(pred_mask, true_mask, smooth=1e-6):
    """
    Вычисление Recall на уровне пикселей
    
    Args:
        pred_mask: предсказанная маска (torch.Tensor)
        true_mask: ground truth маска (torch.Tensor)
        smooth: сглаживающий коэффициент
    
    Returns:
        recall: Recall (float)
    """
    # TODO: Реализуйте Recall
    pass

In [ ]:
# ===== SANITY CHECK: Метрики =====

# --- Идеальное совпадение: все метрики = 1 ---
p1 = torch.ones(1, 1, 4, 4)
t1 = torch.ones(1, 1, 4, 4)
assert abs(dice_score(p1, t1)      - 1.0) < 1e-4, f"Dice идеал: ожидали 1.0, получили {dice_score(p1,t1):.4f}"
assert abs(iou_score(p1, t1)       - 1.0) < 1e-4, f"IoU идеал: ожидали 1.0, получили {iou_score(p1,t1):.4f}"
assert abs(pixel_accuracy(p1, t1)  - 1.0) < 1e-4, f"Accuracy идеал: ожидали 1.0, получили {pixel_accuracy(p1,t1):.4f}"
assert abs(pixel_precision(p1, t1) - 1.0) < 1e-4, f"Precision идеал: ожидали 1.0, получили {pixel_precision(p1,t1):.4f}"
assert abs(pixel_recall(p1, t1)    - 1.0) < 1e-4, f"Recall идеал: ожидали 1.0, получили {pixel_recall(p1,t1):.4f}"

# --- Нет пересечения: Dice и IoU ≈ 0 ---
p0 = torch.zeros(1, 1, 4, 4)
t1 = torch.ones(1, 1, 4, 4)
assert dice_score(p0, t1) < 0.01, f"Dice нет пересечения: ожидали ≈0, получили {dice_score(p0,t1):.4f}"
assert iou_score(p0, t1)  < 0.01, f"IoU нет пересечения: ожидали ≈0, получили {iou_score(p0,t1):.4f}"

# --- Все фон: pixel_accuracy = 1 (вот почему accuracy обманчива!) ---
p0 = torch.zeros(1, 1, 4, 4)
t0 = torch.zeros(1, 1, 4, 4)
assert abs(pixel_accuracy(p0, t0) - 1.0) < 1e-4, "Accuracy при пустых масках должна быть 1.0"

# --- Диапазон значений: все метрики в [0, 1] ---
p_rand = (torch.rand(2, 1, 8, 8) > 0.5).float()
t_rand = (torch.rand(2, 1, 8, 8) > 0.5).float()
for name, val in [
    ("dice",      dice_score(p_rand, t_rand)),
    ("iou",       iou_score(p_rand, t_rand)),
    ("accuracy",  pixel_accuracy(p_rand, t_rand)),
    ("precision", pixel_precision(p_rand, t_rand)),
    ("recall",    pixel_recall(p_rand, t_rand)),
]:
    assert 0.0 <= val <= 1.0, f"{name} вышел за [0,1]: {val:.4f}"

print("✅ Все проверки метрик пройдены!")

<a id="unet"></a>
# 8. Архитектура модели: U‑Net

*Если вдруг вы хотите реализовать другу модель - это ок. Но*

```python
import model
model.train()
```
*не засчитывается*

### Вопрос 9. 

> **Почему U-Net подходит для медицинской сегментации?**
>
> **Ответ ИИ-ассистента:** [TODO: Вставьте ответ]
> 
> **Ваш ответ своими словами:** [TODO: Объясните своими словами]


## Описание архитектуры U-Net

U-Net состоит из двух основных частей:

> 1. **Encoder (Сжимающий путь):**
   - Последовательность сверточных слоев и max pooling
   - Извлекает признаки и уменьшает пространственное разрешение
   - Увеличивает количество каналов

> 2. **Decoder (Расширяющий путь):**
   - Последовательность upsampling и сверточных слоев
   - Восстанавливает пространственное разрешение
   - Уменьшает количество каналов

> 3. **Skip Connections:**
   - Соединяют соответствующие уровни encoder и decoder
   - Помогают сохранить детали из исходного изображения

![UNet architecture](https://raw.githubusercontent.com/mryab/dl-hse-ami/master/week06_vision/UNet.png)


### Объяснение ключевых решений:

> 1. **Почему BatchNorm после каждой свертки?**
   - Стабилизирует обучение
   - Позволяет использовать более высокий learning rate
   - Уменьшает internal covariate shift

> 2. **Почему ReLU activation?**
   - Простая и эффективная нелинейность
   - Помогает избежать vanishing gradient problem
   - Вычислительно эффективна

> 3. **Почему skip connections?**
   - Сохраняют детали из исходного изображения
   - Помогают градиентам распространяться
   - Улучшают локализацию границ объектов

### Полезные материалы для углубленного понимания:

*   **Оригинальная статья:** [U-Net: Convolutional Networks for Biomedical Image Segmentation](https://arxiv.org/abs/1505.04597) - мастрид.
*   **Отличная визуализация архитектуры:** [U-Net explanaible.ai](https://www.explainable.ai/u-net) (интерактивная и очень наглядная).
*   **Разбор архитектуры на пальцах:** множество статей на Medium и Habr, например, [Understanding U-Net](https://towardsdatascience.com/understanding-u-net-61276b10f360).

In [ ]:
class DoubleConv(nn.Module):
    """
    Двойной сверточный блок: (Conv2d -> BatchNorm -> ReLU) x 2
    
    Используется как базовый строительный блок в U-Net
    """
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """
    Downscaling блок: MaxPool -> DoubleConv
    
    Уменьшает пространственное разрешение в 2 раза
    Увеличивает количество каналов
    """
    def __init__(self, in_channels, out_channels):
        # TODO: Реализуйте метод
        pass

    def forward(self, x):
        # TODO: Реализуйте метод
        pass

class Up(nn.Module):
    """
    Upscaling блок: Upsample -> Concatenate -> DoubleConv
    
    Увеличивает пространственное разрешение в 2 раза
    Объединяет с skip connection из encoder
    Уменьшает количество каналов
    """
    def __init__(self, in_channels, out_channels):
        # TODO: Реализуйте метод
        pass

    def forward(self, x1, x2):
        # TODO: Реализуйте метод
        pass

        """
        Args:
            x1: вход из предыдущего слоя decoder
            x2: skip connection из encoder
        """

class UNet(nn.Module):
    """
    U-Net архитектура для сегментации изображений
    
    Args:
        n_channels: количество входных каналов (1 для grayscale, 3 для RGB)
        n_classes: количество выходных классов (1 для бинарной сегментации)
    """
    def __init__(self, n_channels, n_classes):
        # TODO: Реализуйте метод
        pass


    def forward(self, x):
        # TODO: Реализуйте метод
        pass

        # Encoder
        # Decoder with skip connections
        # Output

In [ ]:
# Создание модели
# Для BraTS (один канал MRI): n_channels=1
# Для ISIC (RGB изображения): n_channels=3
# Для бинарной сегментации: n_classes=1

model = UNet(n_channels=3, n_classes=1)  # Пример для ISIC
model = model.to(device)
print(f'Количество параметров модели: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ===== SANITY CHECK: UNet =====
model.eval()
dummy_input = torch.randn(2, 3, 256, 256).to(device)

with torch.no_grad():
    dummy_output = model(dummy_input)

assert dummy_output.shape == (2, 1, 256, 256), \
    f"Выход U-Net должен быть (2,1,256,256), получили {tuple(dummy_output.shape)}"

# Проверяем, что выход — логиты (не прошедшие sigmoid), т.е. есть значения < 0 или > 1
assert dummy_output.min().item() < 0 or dummy_output.max().item() > 1, \
    "Выход похоже уже прошёл sigmoid — forward должен возвращать логиты, а не вероятности!"

# Проверяем, что градиенты текут через модель
dummy_input_grad = torch.randn(1, 3, 256, 256, requires_grad=True).to(device)
out = model(dummy_input_grad)
out.sum().backward()
assert dummy_input_grad.grad is not None, "Градиент не вычислился — проверьте forward!"

print(f"✅ UNet: input={tuple(dummy_input.shape)} → output={tuple(dummy_output.shape)}")

<a id="loss"></a>
# 9. Функция потерь для сегментации


## Dice Loss - рекомендуемая функция потерь

**Почему Dice Loss?**

> 1. **Работает с дисбалансом классов:**
   - В медицинской сегментации часто область интереса занимает малую часть изображения
   - Dice Loss фокусируется на overlap, а не на всех пикселях

> 2. **Напрямую оптимизирует целевую метрику:**
   - Мы оцениваем модель по Dice Score
   - Логично оптимизировать ту же метрику

> 3. **Дифференцируема:**
   - Можно использовать для обучения с gradient descent


$$
\text{Dice Loss} = 1 - \text{Dice Score} = 1 - \frac{2 \times |X \cap Y| + \epsilon}{|X| + |Y| + \epsilon}
$$

Где:
- $X$ - предсказанная маска (после sigmoid)
- $Y$ - ground truth маска
- $\epsilon$ - малое число для численной стабильности (smooth parameter)

### Вопрос 10

> **Какую функцию потерь можно было бы использовать в данном кейсе помимо Dice Loss и какие могут быть минусы?**
>
> **Ответ ИИ-ассистента:** [TODO: Вставьте ответ]
>
> **Ваш ответ своими словами:** [TODO: Объясните своими словами]


### 💡 Почему мы используем сигмоиду в Dice Loss?

Обратите внимание: в классе `DiceLoss` мы предполагаем, что на вход приходят **логиты** (сырые выходы модели без активации). Почему?

1.  **Численная стабильность:** Объединение sigmoid и loss в одном слое (`nn.BCEWithLogitsLoss`) численно стабильнее, чем раздельная сигмоида + `nn.BCELoss`.
2.  **Гибкость:** Мы можем применять порог (например, 0.5) уже после вычисления loss, во время валидации.
3.  **Совместимость:** Большинство реализаций в PyTorch ожидают логиты.

Поэтому в `forward` мы **НЕ** применяем sigmoid к `pred`, а работаем с raw logits. При вычислении Dice Score для метрик мы уже применяем sigmoid и порог.

In [ ]:
class DiceLoss(nn.Module):
    """
    Dice Loss для бинарной сегментации
    
    Dice Loss = 1 - Dice Score
    Минимизация Dice Loss эквивалентна максимизации Dice Score
    """
    def __init__(self, smooth=1e-6):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, pred, target):
        """
        Args:
            pred: предсказания модели (logits), shape (B, 1, H, W)
            target: ground truth маски, shape (B, 1, H, W)
        
        Returns:
            loss: Dice Loss (scalar)
        """
        # 1. Применить сигмоиду к pred, чтобы получить вероятности (B, 1, H, W)
        # 2. Привести pred и target к виду (B, -1), «сплющив» пространственные размеры
        # 3. Посчитать пересечение
        # 4. Посчитать суммы предсказаний и таргетов по размеру признаков
        # 5. Вычислить dice для каждого элемента батча по формуле:
        #    dice_i = (2 * intersection_i + smooth) / (sum_pred_i + sum_target_i + smooth)
        # 6. Усреднить dice по батчу и вернуть 1 - mean_dice как итоговый лосс
        
        pass
        
# Альтернатива: комбинированная функция потерь
class DiceBCELoss(nn.Module):
    """
    Комбинация Dice Loss и Binary Cross Entropy Loss
    
    Преимущества:
    - BCE помогает с обучением на ранних этапах
    - Dice Loss фокусируется на overlap
    - Вместе дают более стабильное обучение
    """
    def __init__(self, smooth=1e-6, dice_weight=0.5, bce_weight=0.5):
        super(DiceBCELoss, self).__init__()
        self.smooth = smooth
        self.dice_weight = dice_weight
        self.bce_weight = bce_weight
        self.bce = nn.BCEWithLogitsLoss()
    
    def forward(self, pred, target):
        """
        Args:
            pred: предсказания модели (logits), shape (B, 1, H, W)
            target: ground truth маски, shape (B, 1, H, W)
        
        Returns:
            loss: комбинированная loss (scalar)
        """
        # 1. Посчитать BCE‑лосс, используя логиты pred и таргет target:
        #    bce_loss = self.bce(pred, target)
        # 2. Посчитать Dice‑лосс по той же схеме, что и в DiceLoss.forward:
        #    - sigmoid(pred) -> вероятности
        #    - привести pred и target к (B, -1)
        #    - посчитать intersection, суммы и dice по батчу
        #    - dice_loss = 1 - mean_dice
        # 3. Скомбинировать лоссы:
        #    total_loss = self.dice_weight * dice_loss + self.bce_weight * bce_loss
        # 4. Вернуть total_loss
        pass


In [ ]:
# Выбор функции потерь
criterion = DiceLoss()  # Или DiceBCELoss()
print('Функция потерь создана')

In [ ]:
# ===== SANITY CHECK: DiceLoss =====
_criterion = DiceLoss()

# Идеальное предсказание: большие положительные логиты → sigmoid ≈ 1 → loss ≈ 0
pred_good   = torch.full((2, 1, 4, 4), 10.0)
target_ones = torch.ones(2, 1, 4, 4)
loss_good   = _criterion(pred_good, target_ones)
assert loss_good.item() < 0.05, \
    f"Loss при идеальном предсказании должен быть ≈0, получили {loss_good.item():.4f}"

# Плохое предсказание: большие отрицательные логиты → sigmoid ≈ 0 → loss ≈ 1
pred_bad  = torch.full((2, 1, 4, 4), -10.0)
loss_bad  = _criterion(pred_bad, target_ones)
assert loss_bad.item() > 0.95, \
    f"Loss при плохом предсказании должен быть ≈1, получили {loss_bad.item():.4f}"

# Loss должен быть дифференцируемым
pred_grad = torch.randn(2, 1, 4, 4, requires_grad=True)
loss_grad = _criterion(pred_grad, target_ones)
loss_grad.backward()
assert pred_grad.grad is not None, "Градиент не вычислился — loss не дифференцируем!"

print("✅ DiceLoss: все проверки пройдены!")

<a id="training"></a>
# 10. Обучение модели

**Что должно быть сделано в этом разделе (Definition of done):**

Минимально ожидается, что вы:
1. Задаёте гиперпараметры обучения (learning rate, число эпох, batch size, weight decay и т.п.) и кратко обосновываете выбор.
2. Создаёте модель, функцию потерь, оптимизатор (и, опционально, scheduler).
3. Реализуете функции `train_epoch` и `validate_epoch`, которые:
   - корректно переключают режимы модели (`train()` / `eval()`),
   - переносят данные на `device`,
   - считают loss и метрики (минимум Dice, желательно ещё IoU),
   - возвращают **усреднённые по эпохе** значения.
4. Реализуете основной цикл обучения:
   - логируете значения loss/метрик по эпохам (в `history` и/или MLflow/Wandb),
   - сохраняете **лучшую модель по val Dice** в `best_model.pth`,
   - выводите на экран основные метрики по каждой эпохе.
5. В конце выводите лучший достигнутый Dice/IoU и (если хотите) строите графики обучения.


Следует интегрировать логирование (MLflow или Weights & Biases), чтобы удобно отслеживать метрики и графики.

## 10.1 Настройка обучения

In [ ]:
# Гиперпараметры (как пример)
LEARNING_RATE = 1e-4  # Почему 1e-4? Это стандартный старт для Adam.
                      # Если loss не падает, можно уменьшить до 1e-5.
NUM_EPOCHS = 20       # Если метрики на валидации растут к 20-й эпохе, стоит увеличить.
WEIGHT_DECAY = 1e-5   # Легкая L2-регуляризация для борьбы с переобучением.

# Создание модели
model = UNet(n_channels=3, n_classes=1)  # Измените n_channels для вашего датасета
model = model.to(device)

# Функция потерь
criterion = DiceLoss()  # или DiceBCELoss()

# Оптимизатор (выберите)
# optimizer = 

# Learning rate scheduler (выберите)
# scheduler =  

print(f'Модель создана. Параметров: {sum(p.numel() for p in model.parameters()):,}')

## 10.2 Настройка логирования экспериментов 
Очень полезная штука для работы - когда вы запускаете долгие/большие экспы, то не очень хочется постоянно трекать все в IDE, поэтому существуют сайты для логгирования обучения где можно смотреть красивые графики. Вот тело-пример настройки логгирования + мануалки как это настроить. По итогу хочется увидеть от вас графики обучения через один из двух сервисов

### Вариант A: MLflow 

> https://mlflow.org/docs/latest/ml/tracking/quickstart/

In [ ]:
import mlflow
import mlflow.pytorch

# Настройка MLflow
mlflow.set_experiment('medical_segmentation')

# Начало эксперимента
with mlflow.start_run(run_name='unet_baseline'):
    # TODO: Логирование гиперпараметров
    
    # TODO: Во время обучения: функция ошибки на обучении и валидации каждую эпоху
    # TODO После обучения: логируем модель

### Вариант B: Weights & Biases (Wandb) 

> https://docs.wandb.ai/models/quickstart

In [ ]:
import wandb

# Инициализация wandb
wandb.init(
    project='medical-segmentation',
    name='unet-baseline',
    config={
        'model': 'UNet',
        'learning_rate': LEARNING_RATE,
        'batch_size': BATCH_SIZE,
        'num_epochs': NUM_EPOCHS,
        'optimizer': 'Adam',
        'loss': 'DiceLoss'
    }
)

# Во время обучения:
# wandb.log({'train_loss': loss_value, 'val_dice': dice_value, 'epoch': epoch})
# 
# Логирование изображений:
# wandb.log({'predictions': wandb.Image(image_with_mask)})
# 
# После обучения:
# wandb.finish()

## 10.3 Функции для обучения и валидации

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    """
    Одна эпоха обучения
    
    Returns:
        avg_loss: средний loss
        avg_dice: средний Dice Score
    """
    # TODO: Реализуйте функцию
    # 1. Переведите модель в нужный режим (train/eval)
    # 2. Создайте цикл по DataLoader с tqdm
    # 3. Переместите данные на device
    # 4. Forward pass
    # 5. Вычислите loss
    # 6. Backward pass: optimizer.zero_grad(), loss.backward(), optimizer.step()
    # 7. Вычислите метрики (dice, iou)
    # 8. Накапливайте значения
    # 9. Верните средние значения
    pass

## 10.4 Основной цикл обучения

In [ ]:
# TODO: Реализуйте основной цикл обучения

best_dice = 0.0
history = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}

for epoch in range(NUM_EPOCHS):
    print(f'\nЭпоха {epoch+1}/{NUM_EPOCHS}')
    
    train_loss, train_dice = train_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice, val_iou = validate_epoch(model, val_loader, criterion, device)
    
    # TODO 
    
    # Логирование (выберите MLflow или Wandb)
    # mlflow.log_metrics({...}, step=epoch)
    
    # wandb.log({...})
    
    # Сохранение лучшей модели
    if val_dice > best_dice:
        best_dice = val_dice
        torch.save(model.state_dict(), 'best_model.pth')
        print(f' Сохранена лучшая модель с Dice={val_dice:.4f}')
    
    print(f'Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}, Val IoU: {val_iou:.4f}')
    
    if epoch == NUM_EPOCHS - 1:
        if val_dice < 0.60:
            print('⚠️ ВНИМАНИЕ: Dice Score < 0.60. Требования не выполнены!')
        if val_iou < 0.50:
            print('⚠️ ВНИМАНИЕ: IoU < 0.50. Требования не выполнены!')

print(f'\nОбучение завершено. Лучший Dice Score: {best_dice:.4f}')

## 10.5 Визуализация процесса обучения

Здесь вы строите графики:

- train / val loss по эпохам,
- train / val Dice, val IoU по эпохам.

По этим графикам важно сделать короткий комментарий:

- есть ли переобучение;
- как быстро модель сходится;
- помогает ли scheduler (если вы его используете).

In [ ]:
# TODO: Постройте графики обучения

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss')
axes[0].plot(history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True)

# Metrics
axes[1].plot(history['train_dice'], label='Train Dice')
axes[1].plot(history['val_dice'], label='Val Dice')
axes[1].plot(history['val_iou'], label='Val IoU')
axes[1].axhline(y=0.60, color='r', linestyle='--', label='Min Dice (0.60)')
axes[1].axhline(y=0.50, color='orange', linestyle='--', label='Min IoU (0.50)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Score')
axes[1].set_title('Training and Validation Metrics')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

<a id="evaluation"></a>
# 11. Оценка и визуализация результатов

На этом этапе у вас уже есть обученная модель и сохранённый чекпоинт `best_model.pth`. Здесь важно:

1. **Численно** оценить модель на валидации (метрики).
2. **Визуально** посмотреть на примеры:
   - хорошие предсказания (высокий Dice),
   - проблемные случаи (ложно положительные / ложно отрицательные).
3. Связать результаты с медицинским контекстом.

**Что должно быть сделано (Definition of done):**

1. Загрузить лучшую модель (`best_model.pth`) и посчитать итоговые метрики на валидации:
   - средний Dice Score,
   - средний IoU,
   - при желании Pixel Accuracy, Precision, Recall.
2. Построить минимум **6 визуализаций**:
   - 3 примера с хорошим качеством (высокий Dice, IoU),
   - 3 примера с ошибками (низкий Dice, пропущенные/лишние участки).
   На каждой визуализации показывать **трёхпанельный вид**:
   - исходное изображение,
   - ground truth маска,
   - предсказанная моделью маска (можно overlay).
3. Написать 1–2 абзаца аналитики:
   - какие типичные ошибки делает модель (недосегментация / пересегментация; зависимость от размера или контраста поражения),
   - есть ли связь между ошибками и статистикой из EDA (например, маленькие объекты, артефакты).

Кратко прокомментируйте, достигли ли вы порогов из условий ДЗ и удовлетворены ли качеством.

### 🧐 Как анализировать визуализации?

При просмотре примеров обращайте внимание на:

1.  **Границы объекта:** Модель четко обводит контур или "размазывает" его?
2.  **Маленькие объекты:** Видит ли модель маленькие поражения/опухоли или пропускает их?
3.  **Связные области:** Есть ли "дырки" внутри предсказанной маски там, где объект должен быть сплошным?
4.  **Ложные срабатывания:** Где модель ошибочно находит объекты (на границах кадра, на артефактах)?


## 11.1 Финальная оценка на валидационной выборке

In [ ]:
# TODO: Загрузите лучшую модель и оцените на валидационной выборке

# Загрузка лучшей модели
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

# Финальная оценка
final_loss, final_dice, final_iou = validate_epoch(model, val_loader, criterion, device)

print(f'Loss: {final_loss:.4f}')
print(f'Dice Score: {final_dice:.4f}')
print(f'IoU Score: {final_iou:.4f}')
print('='*50)

# Проверка требований
if final_dice >= 0.60 and final_iou >= 0.50:
    print('Минимальные требования выполнены!')
else:
    print('Минимальные требования НЕ выполнены!')
    if final_dice < 0.60:
        print(f'   - Dice Score {final_dice:.4f} < 0.60')
    if final_iou < 0.50:
        print(f'   - IoU Score {final_iou:.4f} < 0.50')

## 11.2 Качественная оценка

Покажите несколько примеров:

- где модель сегментирует очень хорошо (визуально);
- где модель явно ошибается.

Обязательно подпишите, что именно вы считаете «хорошим» и «плохим» примером.

In [ ]:
# TODO: Визуализируйте лучшие и худшие примеры

def visualize_predictions(model, loader, device, n_samples=10, best=True):
    """
    Визуализация предсказаний модели
    Args:
        model: обученная модель
        loader: DataLoader
        device: устройство
        n_samples: количество примеров
        best: True для лучших примеров, False для худших
    """
    model.eval()
    samples = []
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            preds = torch.sigmoid(outputs) > 0.5
            
            for i in range(len(images)):
                dice = dice_score(preds[i:i+1], masks[i:i+1])
                samples.append({
                    'image': images[i].cpu(),
                    'mask': masks[i].cpu(),
                    'pred': preds[i].cpu(),
                    'dice': dice
                })
    
    # Сортируем по Dice
    samples.sort(key=lambda x: x['dice'], reverse=best)
    samples = samples[:n_samples]
    
    # Визуализация
    fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4*n_samples))
    
    for i, sample in enumerate(samples):
        # Изображение
        img = sample['image'].permute(1, 2, 0).numpy()
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f'Image (Dice={sample["dice"]:.3f})')
        axes[i, 0].axis('off')
        
        # Ground Truth
        axes[i, 1].imshow(sample['mask'].squeeze(), cmap='gray')
        axes[i, 1].set_title('Ground Truth')
        axes[i, 1].axis('off')
        
        # Prediction
        axes[i, 2].imshow(sample['pred'].squeeze(), cmap='gray')
        axes[i, 2].set_title('Prediction')
        axes[i, 2].axis('off')
        
        # Overlay
        axes[i, 3].imshow(img)
        axes[i, 3].imshow(sample['pred'].squeeze(), alpha=0.5, cmap='Reds')
        axes[i, 3].set_title('Overlay')
        axes[i, 3].axis('off')
    
    plt.tight_layout()
    plt.show()

print('Лучшие примеры:')
visualize_predictions(model, val_loader, device, n_samples=5, best=True)

print('Худшие примеры:')
visualize_predictions(model, val_loader, device, n_samples=5, best=False)

## 11.3 Анализ ошибок

**Задание:**

- Опишите типичные **False Positives** (где модель видит объект там, где его нет). Связаны ли они с артефактами, которые вы заметили в EDA (волосы, пузырьки, череп)?
- Опишите типичные **False Negatives** (где модель пропускает патологию). Связаны ли они с размером объекта, его контрастностью или положением на изображении?
- Отметьте проблемы на **границах объекта**: модель рисует слишком "пухлые" или "тощие" маски?
- Проанализируйте **связность предсказаний**: есть ли "дырки" там, где их быть не должно?

**Структура ответа (рекомендуемая):**

1.  **Статистика:** приведите примерные цифры (например, "в 30% ошибок модель путает волосы с поражением").
2.  **Примеры:** сошлитесь на конкретные визуализации из предыдущего пункта ("на картинке #3 видно, что...").
3.  **Гипотезы:** почему так происходит? (не хватает аугментаций с волосами, дисбаланс классов, маленький receptive field).
4.  **Пути решения:** что можно попробовать изменить в пайплайне, чтобы исправить эти ошибки?

**Ваш анализ:**

[TODO: опишите своими словами типичные ошибки и возможные пути улучшения (другие аугментации, баланс классов, изменение loss и т.п.)]


<a id="interpretability"></a>
# 12. Интерпретация / Объяснимость (XAI)

В медицинской сегментации мало получить хорошие метрики (Dice/IoU). Врачу критически важно понимать, **почему** модель приняла то или иное решение. Если модель выделяет невус, ориентируясь на черную рамку дерматоскопа или линейку на коже, — такая модель неприменима в клинике, даже если у нее отличный Dice Score.

Здесь вы:
- используете библиотеку **Captum** (от PyTorch) для интерпретации предсказаний;
- примените выбранный метод к нескольким примерам из валидации (как к успешным, так и к ошибочным);
- визуализируете карты атрибуции (важности пикселей).

Важно не просто сгенерировать цветную картинку, но и **коротко прокомментировать** её:
- совпадает ли фокус модели с клинически значимыми областями (опухоль / поражение кожи);
- видите ли вы «подозрительные» паттерны (модель реагирует на артефакты: маркеры, волосы, границы кадра).

---

### Краткая справка: как работать с Captum для сегментации

В отличие от классификации, где выход модели — один логит, в сегментации выход — это пространственная карта (B, C, H, W). Большинство XAI-методов (например, `IntegratedGradients` или `Saliency`) ожидают на выходе скаляр. 

Чтобы обойти это, в Captum обычно используют **целевую функцию (target)**, которая агрегирует нужные пиксели предсказания (например, сумму логитов в области предсказанной маски).

**Установка:**
```bash
!pip install captum
```

**Доступные методы в Captum, которые хорошо подходят для картинок:**
1. `Saliency` — самый базовый метод (градиенты выхода по входу, **его не берем**).
2. `IntegratedGradients` (IG) — классический бейзлайн.
3. `GuidedGradCam` — GradCam + GuidedBackpropagation, highly recommend
4. `LayerGradCam` — популярный метод, применяемый к последнему сверточному слою энкодера.

### Вопрос 11. Выбор метода XAI

> **Задание:**
> - Выберите ДВА метода интерпретации из Captum (или реализуйте свой, например, Grad-CAM).
> - Объясните в 2-3 предложениях, как математически/концептуально работают выбранные алгоритмы.
> - Почему эти методы применимы к задаче сегментации?
>
> **Ответ ИИ-ассистента:** [TODO: Вставьте ответ]
>
> **Ваш ответ своими словами:** [TODO: Перефразируйте и обоснуйте выбор метода]

### 💡 Важное замечание по реализации:

Приведенный ниже код с Saliency и `custom_forward`, суммирующей выход, является **упрощенным примером**. На практике такой подход может давать зашумленные карты. Для более качественных результатов рекомендуется:

1.  Использовать `LayerGradCam` на последнем слое энкодера (перед "бутылочным горлышком").
2.  Использовать `Occlusion`, если позволяет вычислительная мощность, так как он дает очень интуитивные карты, показывая, как закрытие части изображения влияет на предсказание.
3.  Для `IntegratedGradients` в качестве `baseline` часто используют черное или шумное изображение.

Экспериментируйте! Главное — понять принцип.

In [ ]:
# !pip install captum

import torch
from captum.attr import # TODO
from captum.attr import visualization as viz

# TODO: Реализуйте функцию визуализации с помощью выбранного метода
# Ниже приведен СТАРТОВЫЙ ШАБЛОН для Saliency, чтобы показать принцип работы с Captum.
# Вы можете изменить его на LayerGradCam, IntegratedGradients или любые другие релевантные.

def get_attribution_map(model, image_tensor, target_mask=None):
    """
    Генерирует карту атрибуции для одного изображения.
    
    Args:
        model: обученная модель сегментации (в режиме eval)
        image_tensor: тензор изображения (1, C, H, W) с requires_grad=True
        target_mask: опционально, маска пикселей, по которым мы считаем важность
    """
    model.eval()
    image_tensor.requires_grad = True
    
    # 1. Инициализация метода (Пример с Saliency)
    saliency = Saliency(model)
    
    # Для сегментации нам нужно свести выход (1, 1, H, W) к скаляру.
    # Самый простой способ (не всегда лучший) - взять сумму всех положительных логитов.
    # В Captum это можно сделать через кастомную forward функцию:
    
    def custom_forward(inputs):
        outputs = model(inputs) # (1, 1, H, W)
        # Суммируем логиты там, где модель предсказывает класс 1
        # Или просто суммируем весь выход
        return outputs.sum(dim=(2, 3))
    
    # Инициализируем метод с custom_forward
    saliency_custom = Saliency(custom_forward)
    
    # 2. Вычисление атрибуций
    attributions = saliency_custom.attribute(image_tensor)
    
    # Усредняем по каналам (если RGB) для визуализации
    if attributions.shape > 1:
        attributions = attributions.mean(dim=1, keepdim=True)
        
    return attributions.detach().cpu()

# TODO: Сгенерируйте и визуализируйте карты для:
# - минимум 3 правильных случаев (высокий Dice)
# - минимум 3 случаев с ошибками (низкий Dice, False Positives/Negatives)

# Подсказка по визуализации:
# Можно использовать стандартный plt.imshow(attributions) или встроенный инструмент Captum:
# viz.visualize_image_attr(
#     np.transpose(attributions.squeeze().cpu().numpy()[..., np.newaxis], (0, 1, 2)),
#     np.transpose(original_image.squeeze().cpu().numpy(), (1, 2, 0)),
#     method="blended_heat_map",
#     sign="all",
#     show_colorbar=True
# )

### Вопрос 12. Анализ карт важности

Опираясь на полученные визуализации из ячейки выше, дайте развернутый ответ на следующие вопросы.

> **Задание: Интерпретация результатов**
> 1. **Куда «смотрит» модель?** (Совпадают ли яркие пиксели на картах атрибуции с реальным расположением невуса/опухоли?)
> 2. **Клинически ли это правдоподобно?** (Учитывает ли модель текстуру самой патологии, или она смотрит только на резкие границы и цвет кожи вокруг?)
> 3. **Есть ли артефакты/смещения?** (Посмотрите на примеры с ошибками. Реагирует ли ваша модель на волосы (для ISIC), пузырьки воздуха, края кадра, маркеры врачей или череп (для BraTS)?)
> 4. **Различия между задачами** (Если вы пробовали смотреть на разные классы или типы ошибок: отличаются ли паттерны внимания при False Positive и False Negative?)
>
> **Ваш ответ:** 
> [TODO: Напишите связный аналитический текст (3-5 абзацев), ссылаясь на конкретные визуализированные примеры. Например: "На картинке №2 с низким Dice видно, что градиенты сфокусировались на волосе, пересекающем родинку, из-за чего модель...". Сравните выходы двух методов]

### Самое важное (и немного отдыха) 🧘

Поздравляем! Вы прошли через все круги нашего ДЗ: разбирались со структурой данных, отлаживали U-Net, боролись с переобучением и интерпретировали результаты. Вы заслужили немного творчества.

*   Сгенерируйте картинку (Midjourney, DALL-E, Kandinsky — любым инструментом), иллюстрирующую, какие эмоции вы испытали в процессе работы над этим ДЗ.
*   Или просто пришлите что-то очень красивое/смешное/странное, что нейросеть создала специально для вас.

🎁 **Бонусные очки за креативность:** если ваша картинка будет связана с медицинской сегментацией (например, U-Net, рисующий контуры вокруг кота, или МРТ-сканер, генерирующий мемы).

**Место для вашего шедевра:**

<a id="bonus"></a>
# 13. Бонус: использование предобученных моделей

В бонусной части вы можете:

- взять предобученную модель (например, из `torchvision` или специализированной библиотеки);
- адаптировать её к своей задаче (fine‑tuning, изменение головы);
- сравнить её с вашей «ручной» U‑Net по метрикам и визуально.

**Важно:** просто `import model; model.train()` без понимания внутренней структуры **не засчитывается** как основное решение. Бонус — это надстройка над вашей базовой реализации, а не замена.

## Важные ограничения

- Если предобученная модель была **обучена напрямую на BraTS, ISIC, или их официальных challenge датасетах**, вы **не должны её использовать (если вдруг не можете найти такой инфы - забейте и используйте, что нашли**.
- В этом случае выберите **другую модель**, которая:
  - максимально близка архитектурно, и
  - предобучена на **другом домене** (например, общие медицинские изображения).

- Если **не существует предобученной версии** для вашей архитектуры:
  - выберите **ближайшую доступную архитектуру** с предобученными весами,
  - четко объясните архитектурные различия.

Все выборы должны быть явно задокументированы.

### 📚 Где брать предобученные модели?

1.  **[PyTorch Hub](https://pytorch.org/hub/):** Огромный репозиторий предобученных моделей.
2.  **[timm](https://github.com/rwightman/pytorch-image-models):** Библиотека Ross Wightman с сотнями предобученных энкодеров (ResNet, EfficientNet, ViT и др.).
3.  **[segmentation_models_pytorch](https://github.com/qubvel/segmentation_models.pytorch):** Специализированная библиотека для сегментации с предобученными энкодерами (⚠️ только для бонуса, не для основной части!).
4.  **[Hugging Face Models](https://huggingface.co/models):** Можно найти модели для медицинской сегментации (например, ViT для медицины).

In [ ]:
# TODO: Реализуйте эксперимент с предобученной моделью

# 1. Загрузите предобученную модель или encoder
# 2. Адаптируйте для вашей задачи (измените входные/выходные слои если нужно)
# 3. Обучите на вашем датасете
# 4. Сравните с обучением с нуля

pass

## Отчет и сравнение

Для обоих вариантов обучения отчитайтесь:

- скорость сходимости обучения (графики loss / метрик),
- финальные метрики оценки,
- качественные различия в предсказаниях и случаях ошибок,
- сравнение интерпретаций (тот же pipeline как выше)

**Ваш отчет:** [TODO: Подробный отчет о сравнении]

### Вопрос 13 (бонус)

> **Ответьте на следующие вопросы:**
>   1. Что такое **pretraining** в deep learning?
>   2. Почему pretraining может помочь обучению модели?
>   3. Может ли pretraining иногда **навредить** производительности?
>   4. Наблюдаете ли вы случаи, когда pretraining **не полезен** для вашего датасета?
>   5. Как различается эффект pretraining между MRI данными и дерматоскопическими изображениями?
>
>**Ответ ИИ-ассистента:** [TODO: Вставьте ответ]
>
> **Ваш ответ своими словами:** [TODO: Перефразируйте и ответьте на все вопросы]